In [87]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
import uuid
import pydantic_settings
import chromadb
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [88]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    for pdf_file in pdf_files:
        print(f"processing {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            pages = loader.load()
            for page in pages:
                page.metadata['sourcefile'] = pdf_file.name
                page.metadata['file_type'] = 'pdf'
            all_documents.extend(pages)
            print(f"loaded {len(pages)} pages")
        except Exception as e:
            print(f"error is {e}")
    return all_documents

all_pdf_documents = process_all_pdfs(".")



Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)


processing uganda tourism.pdf
loaded 2 pages


In [89]:
len(all_pdf_documents)

2

In [90]:
# split text into chunks
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,
                                                   chunk_overlap = chunk_overlap,
                                                   length_function = len,
                                                   separators = ["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")
    
    return split_docs


    

In [91]:
split1 = split_documents(all_pdf_documents)


split 2 documents into 4 chunks


In [92]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-V2"):
        self.model_name = model_name
        self.model  = None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded. Embedding dimensons {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Encountered error {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts)
        print(f"generated embeddings with shape {embeddings.shape}")
        return embeddings
    

embedding_manager = EmbeddingManager()
embedding_manager


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: a380492b-6a5e-44e7-9ae1-ed4e37cc9bd5)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-V2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading all-MiniLM-L6-V2
Model loaded. Embedding dimensons 384


In [93]:
# vectorstore db
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", presist_directory = "vector_store"):
        self.collection_name = collection_name
        self.persist_directory = presist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            self.collection = self.client.get_or_create_collection(name = self.collection_name,
                                                                   metadata= {"description": "pdf document embeddings"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Exisiting documents in collection {self.collection.count()}")
        except Exception as e:
            print(f"Error encounterd: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.array):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents not equal to number of embeddings")
        print(f"adding {len(documents)} to vector store")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"successifuly added {len(documents)} to vector db")
            print(f"total documents in collection are {self.collection.count()}")
        except Exception as e:
            print(f"error in adding doc {e}")
            raise

vectorstore = VectorStore()

Vector store initialized. Collection: pdf_documents
Exisiting documents in collection 12


In [94]:
texts = [doc.page_content for doc in split1]

#generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in vector db
vectorstore.add_documents(split1,embeddings)


generating embeddings for 4 texts
generated embeddings with shape (4, 384)
adding 4 to vector store
successifuly added 4 to vector db
total documents in collection are 16


In [104]:
class RAGretriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str,Any]]:
        print(f"Retreiving documetns for {query}")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        #query embedding
        query_emebdding = self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vector_store.collection.query(query_embeddings=[query_emebdding.tolist()],
                                                         n_results=top_k)
            
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results["ids"][0]

                for i ,(doc_id, document, metadata,distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({'id':doc_id,
                                               'content': document,
                                               'metadata': metadata,
                                               'similarity score': similarity_score,
                                               'distance': distance,
                                               'rank': i + 1})
                        
                print(f"Retrieved {len(retrieved_docs)} docs after filtering")
                return retrieved_docs
            else:
                print("no documents found")
        except Exception as e:
            print(f"error during retrieval {e}")
   
            return []

In [105]:
rag_retriever = RAGretriever(vectorstore, embedding_manager)

In [106]:
rag_retriever

In [107]:
rag_retriever.retrieve("where is kampala")

Retreiving documetns for where is kampala
Top K: 5, score threshold: 0.0
generating embeddings for 1 texts
generated embeddings with shape (1, 384)
Retrieved 4 docs after filtering


[{'id': 'doc_0a87c9a2_0',
  'content': 'Document Title: Historical and Cultural Heritage Sites of Uganda Focus: Pre-colonial, Colonial, and Spiritual Landmarks 1. Kasubi Royal Tombs (UNESCO World Heritage Site) • Location: Kampala • Significance: This is the burial ground for four former Kings (Kabakas) of the Buganda Kingdom. The main building, Muzibu-Azaala-Mpanga, is a masterpiece of traditional Ganda architecture, constructed using bamboo, wood, and a thick thatched roof. It represents the living cultural traditions of the Baganda people. 2. Namugongo Martyrs Shrine • Location: Wakiso District (near Kampala) • Significance: This site commemorates 45 Christian converts (Catholic and Anglican) who were executed between 1885 and 1887 on the orders of Kabaka Mwanga II. The Catholic shrine is architecturally unique, built in the shape of a traditional African hut with 22 copper pillars. 3. Nyero Rock Paintings • Location: Kumi District (Eastern Uganda) • Significance: Dating back to the

vector DB context pipeline with llm output

In [114]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
API_KEY = "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"
llm = ChatGroq(groq_api_key = API_KEY, model_name = "llama-3.1-8b-instant", temperature=0.1, max_tokens= 1024)

def rag_simple(query, retriever, llm, top_k = 3):
    results =retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "no relevant context found"
    prompt = f""" Use the following context to answer the question concisely.
    Context: {context},
    question: {query}   
    answer: """
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [115]:
answer = rag_simple("where is kampala", rag_retriever,llm)

Retreiving documetns for where is kampala
Top K: 3, score threshold: 0.0
generating embeddings for 1 texts
generated embeddings with shape (1, 384)
Retrieved 3 docs after filtering


In [116]:
print(answer)

Kampala.


In [ ]:
answer = rag_simple(" ", rag_retriever,llm)

Retreiving documetns for What is the Namugongo Martyrs Shrine
Top K: 3, score threshold: 0.0
generating embeddings for 1 texts
generated embeddings with shape (1, 384)
Retrieved 3 docs after filtering


In [121]:
print(answer)

This site commemorates 45 Christian converts (Catholic and Anglican) who were executed between 1885 and 1887 on the orders of Kabaka Mwanga II. The Catholic shrine is architecturally unique, built in the shape of a traditional African hut with 22 copper pillars.
